In [0]:
from pyspark.sql import functions as F
import uuid
import traceback

BASE_PATH = "abfss://raw@adlsfootballiqdev.dfs.core.windows.net"
TARGET_SCHEMA = "football_catalog.bronze"

tables_to_load = {
    "appearances": "appearances",
    "club_games": "club_games",
    "clubs": "clubs",
    "competitions": "competitions",
    "countries": "countries",
    "game_events": "game_events",
    "game_lineups": "game_lineups",
    "games": "games",
    "national_teams": "national_teams",
    "player_valuations": "player_valuations",
    "players": "players",
    "transfers": "transfers"
}

# Keep track of successes and failures for alerting
summary = {"success": [], "failed": []}

for folder, table in tables_to_load.items():
    source_path = f"{BASE_PATH}/{folder}/{folder}.csv"
    target_table = f"{TARGET_SCHEMA}.{table}"
    source_file = f"{folder}.csv"
    
    print(f"--- Processing: {table} ---")
    
    try:
        # Read raw CSV
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "false")
              .option("multiLine", "true")
              .option("escape", '"')
              .csv(source_path))
        
        # Add metadata
        batch_id = str(uuid.uuid4())
        df_meta = (df
            .withColumn("_load_ts",     F.current_timestamp())
            .withColumn("_source_file", F.lit(source_file))
            .withColumn("_batch_id",    F.lit(batch_id)))
        
        # Write to Unity Catalog (Check if you need append or overwrite!)
        (df_meta.write
         .format("delta")
         .mode("overwrite") # CHANGE TO "append" IF FILES ARE INCREMENTAL
         .option("overwriteSchema", "true")
         .saveAsTable(target_table))
        
        final_count = spark.table(target_table).count()
        print(f"SUCCESS: Written to {target_table} | Total rows: {final_count:,}\n")
        summary["success"].append(table)
        
    except Exception as e:
        print(f"FAILED: Could not process {table}.")
        print(traceback.format_exc())
        summary["failed"].append(table)
        print("\n")

# Print final pipeline status
print("--- PIPELINE SUMMARY ---")
print(f"Successful: {len(summary['success'])} tables")
print(f"Failed: {len(summary['failed'])} tables")
if summary['failed']:
    print(f"Failing tables: {summary['failed']}")
    # You could optionally raise an exception here to fail the Databricks job task